# Лабораторна робота №2 — Частина 1
## Наука про дані: підготовчий етап
### VHI-індекс для адміністративних одиниць України

## 1. Імпорт бібліотек

In [ ]:
import urllib.request
import numpy as np
import pandas as pd
import os
import glob
import re
from datetime import datetime
from io import StringIO

## 2. Завантаження VHI-даних для всіх областей України

Завантажуємо дані VHI-індексу з NOAA для кожної з 27 адміністративних одиниць (provinceID від 1 до 27).
При повторному запуску файли не перезавантажуються — реалізовано захист від дублювання.

In [ ]:
# Папка для збереження даних
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

def download_vhi_data(province_id, year1=1981, year2=2024):
    """
    Завантажує VHI-дані для вказаної області.
    Якщо дані вже завантажені — пропускає.
    До імені файлу додається дата та час завантаження.
    """
    # Перевіряємо чи файл для цієї області вже існує
    existing = glob.glob(os.path.join(DATA_DIR, f"vhi_province_{province_id}_*.csv"))
    if existing:
        print(f"  Область {province_id}: вже завантажено ({os.path.basename(existing[0])}), пропускаємо")
        return existing[0]
    
    url = (f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?"
           f"country=UKR&provinceID={province_id}&year1={year1}&year2={year2}&type=Mean")
    
    now = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = os.path.join(DATA_DIR, f"vhi_province_{province_id}_{now}.csv")
    
    try:
        urllib.request.urlretrieve(url, filename)
        print(f"  Область {province_id}: завантажено -> {os.path.basename(filename)}")
        return filename
    except Exception as e:
        print(f"  Область {province_id}: ПОМИЛКА - {e}")
        return None

# Завантажуємо дані для всіх 27 областей (provinceID=0 — середнє по Україні, не завантажуємо)
print("Завантаження VHI-даних для областей України...")
print("=" * 60)
downloaded_files = {}
for pid in range(1, 28):
    filepath = download_vhi_data(pid)
    if filepath:
        downloaded_files[pid] = filepath

print("=" * 60)
print(f"Завантажено файлів: {len(downloaded_files)}")

## 3. Зчитування та очищення даних (Data Cleaning)

Зчитуємо всі завантажені файли у pandas DataFrame. 
Файли NOAA мають формат: HTML-обгортка `<pre>...</pre>`, всередині — CSV з роздільником `,` та зайвою комою в кінці.
Прибираємо HTML-теги, зайві стовпці, заповнюємо пропуски. Додаємо стовпчики з назвою та індексом області.

In [ ]:
# Маппінг NOAA індексів (англійська абетка) -> назва області
NOAA_ID_TO_NAME = {
    1: "Cherkasy", 2: "Chernihiv", 3: "Chernivtsi", 4: "Crimea",
    5: "Dnipropetrovsk", 6: "Donetsk", 7: "Ivano-Frankivsk",
    8: "Kharkiv", 9: "Kherson", 10: "Khmelnytskyi",
    11: "Kyiv", 12: "Kyiv City", 13: "Kirovohrad", 14: "Luhansk",
    15: "Lviv", 16: "Mykolaiv", 17: "Odessa", 18: "Poltava",
    19: "Rivne", 20: "Sevastopol", 21: "Sumy", 22: "Ternopil",
    23: "Transcarpathia", 24: "Vinnytsia", 25: "Volyn",
    26: "Zaporizhzhia", 27: "Zhytomyr"
}

# Маппінг: українська абетка -> NOAA ID
# Індексація за українською абеткою: 1-Вінницька, 2-Волинська, ...
UKR_INDEX = {
    1: ("Вінницька", 24),
    2: ("Волинська", 25),
    3: ("Дніпропетровська", 5),
    4: ("Донецька", 6),
    5: ("Житомирська", 27),
    6: ("Закарпатська", 23),
    7: ("Запорізька", 26),
    8: ("Івано-Франківська", 7),
    9: ("Київська", 11),
    10: ("Кіровоградська", 13),
    11: ("Кримська", 4),
    12: ("Луганська", 14),
    13: ("Львівська", 15),
    14: ("Миколаївська", 16),
    15: ("Одеська", 17),
    16: ("Полтавська", 18),
    17: ("Рівненська", 19),
    18: ("Сумська", 21),
    19: ("Тернопільська", 22),
    20: ("Харківська", 8),
    21: ("Херсонська", 9),
    22: ("Хмельницька", 10),
    23: ("Черкаська", 1),
    24: ("Чернівецька", 3),
    25: ("Чернігівська", 2),
    26: ("м. Київ", 12),
    27: ("м. Севастополь", 20),
}

# Зворотній маппінг: NOAA ID -> український індекс
NOAA_TO_UKR = {noaa_id: (ukr_idx, name) for ukr_idx, (name, noaa_id) in UKR_INDEX.items()}

print("Маппінг NOAA -> Українська індексація:")
print("-" * 50)
for noaa_id in sorted(NOAA_TO_UKR.keys()):
    ukr_idx, ukr_name = NOAA_TO_UKR[noaa_id]
    print(f"  NOAA {noaa_id:2d} ({NOAA_ID_TO_NAME[noaa_id]:20s}) -> {ukr_idx:2d} {ukr_name}")

In [ ]:
def read_vhi_file(filepath, noaa_province_id):
    try:
        df = pd.read_csv(filepath, skiprows=1, skipinitialspace=True,
                         index_col=False, on_bad_lines='skip',
                         encoding='utf-8', encoding_errors='ignore')
        
        df.columns = [re.sub(r'<[^>]+>', '', c).strip() for c in df.columns]
        rename_map = {'year': 'Year', 'week': 'Week'}
        df = df.rename(columns=rename_map)
        
        for col in df.columns:
            if df[col].dtype == object:
                df[col] = df[col].astype(str).str.replace(r'<[^>]+>', '', regex=True).str.strip()
        
        for col in ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        df = df.replace(-1, np.nan)
        df = df.dropna(subset=['Year', 'Week', 'VCI', 'TCI', 'VHI'])
        
        df['Year'] = df['Year'].astype(int)
        df['Week'] = df['Week'].astype(int)
        
        ukr_idx, ukr_name = NOAA_TO_UKR.get(noaa_province_id, (noaa_province_id, 'Unknown'))
        df['NOAA_ID'] = noaa_province_id
        df['Province_ID'] = ukr_idx
        df['Province_Name'] = ukr_name
        
        return df
    
    except Exception as e:
        print(f'  Помилка: {filepath}: {e}')
        return pd.DataFrame()

print('Зчитування та очищення даних...')
print('=' * 60)

all_data = []
for noaa_id, filepath in sorted(downloaded_files.items()):
    df = read_vhi_file(filepath, noaa_id)
    if not df.empty:
        all_data.append(df)
        print(f'  NOAA {noaa_id:2d}: {len(df)} записів')
    else:
        print(f'  NOAA {noaa_id:2d}: ПОМИЛКА')

vhi_df = pd.concat(all_data, ignore_index=True)

print('=' * 60)
print(f'Загальний DataFrame: {vhi_df.shape[0]} рядків, {vhi_df.shape[1]} стовпців')


In [ ]:
# Перегляд перших записів
print("Перші 10 записів:")
vhi_df.head(10)

In [ ]:
print('Типи даних:')
print(vhi_df.dtypes)
print()
vhi_df.info()
print('\nСтатистика:')
vhi_df.describe()


In [ ]:
print('Кількість пропусків по стовпцях:')
print(vhi_df.isnull().sum())
print(f'\nВідсоток пропусків: {vhi_df.isnull().sum().sum() / vhi_df.size * 100:.2f}%')


## 4. Зміна індексів областей

В завантажених даних NOAA області індексуються за англійською абеткою (Province 1 - Cherkasy). 
Замінюємо на індексацію за українською абеткою (1 - Вінницька, 2 - Волинська, ...).

In [ ]:
# Перевіримо що індекси правильно замінені
print("Унікальні області в DataFrame (українська індексація):")
print("=" * 60)
provinces = vhi_df[['Province_ID', 'Province_Name', 'NOAA_ID']].drop_duplicates().sort_values('Province_ID')
for _, row in provinces.iterrows():
    print(f"  {int(row['Province_ID']):2d}. {row['Province_Name']:25s} (NOAA ID: {int(row['NOAA_ID'])})")

print(f"\nВсього областей: {vhi_df['Province_ID'].nunique()}")

## 5. Функції для формування вибірок

### 5.1. Ряд VHI для області за вказаний рік

In [ ]:
def get_vhi_for_province_year(df, province_id, year):
    """
    Повертає ряд VHI для вказаної області за вказаний рік.
    province_id — індекс за українською абеткою.
    """
    result = df[(df['Province_ID'] == province_id) & (df['Year'] == year)]
    
    if result.empty:
        province_name = UKR_INDEX.get(province_id, ("Невідома",))[0]
        print(f"Даних не знайдено для області {province_id} ({province_name}) за {year} рік")
        return result
    
    province_name = result['Province_Name'].iloc[0]
    print(f"VHI для {province_name} (індекс {province_id}) за {year} рік:")
    print(f"Кількість записів (тижнів): {len(result)}")
    return result[['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'Province_Name']].reset_index(drop=True)

# Приклад: VHI для Київської області (індекс 9) за 2020 рік
get_vhi_for_province_year(vhi_df, 9, 2020)

### 5.2. Ряд VHI за вказаний діапазон років для вказаних областей

In [ ]:
def get_vhi_for_provinces_years(df, province_ids, year_start, year_end):
    """
    Повертає VHI за діапазон років для кількох областей.
    province_ids — список індексів за українською абеткою.
    """
    result = df[
        (df['Province_ID'].isin(province_ids)) & 
        (df['Year'] >= year_start) & 
        (df['Year'] <= year_end)
    ]
    
    if result.empty:
        print(f"Даних не знайдено для областей {province_ids} за {year_start}-{year_end}")
        return result
    
    provinces_found = result['Province_Name'].unique()
    print(f"VHI за {year_start}-{year_end} для областей: {', '.join(provinces_found)}")
    print(f"Кількість записів: {len(result)}")
    return result[['Year', 'Week', 'VHI', 'Province_ID', 'Province_Name']].reset_index(drop=True)

# Приклад: VHI для Львівської (13) та Одеської (15) областей за 2015-2020
get_vhi_for_provinces_years(vhi_df, [13, 15], 2015, 2020)

### 5.3. Пошук екстремумів (min, max), середнього та медіани VHI

In [ ]:
def get_vhi_extremes(df, province_ids, year_start, year_end):
    """
    Знаходить min, max, середнє та медіану VHI для вказаних областей та років.
    """
    result = df[
        (df['Province_ID'].isin(province_ids)) & 
        (df['Year'] >= year_start) & 
        (df['Year'] <= year_end)
    ].copy()
    
    if result.empty:
        print("Даних не знайдено")
        return pd.DataFrame()
    
    print(f"Екстремуми VHI за {year_start}-{year_end}")
    print("=" * 70)
    
    stats_list = []
    for pid in province_ids:
        prov_data = result[result['Province_ID'] == pid]['VHI']
        if prov_data.empty:
            continue
        
        name = UKR_INDEX.get(pid, ("Невідома",))[0]
        stats = {
            'Область': name,
            'ID': pid,
            'Min VHI': prov_data.min(),
            'Max VHI': prov_data.max(),
            'Середнє': round(prov_data.mean(), 2),
            'Медіана': round(prov_data.median(), 2),
            'Кількість': len(prov_data)
        }
        stats_list.append(stats)
        
        # Знаходимо тиждень/рік для min та max
        min_row = result[(result['Province_ID'] == pid) & (result['VHI'] == prov_data.min())].iloc[0]
        max_row = result[(result['Province_ID'] == pid) & (result['VHI'] == prov_data.max())].iloc[0]
        
        print(f"  {name} (ID {pid}):")
        print(f"    Min VHI = {prov_data.min():.2f} (рік {int(min_row['Year'])}, тиждень {int(min_row['Week'])})")
        print(f"    Max VHI = {prov_data.max():.2f} (рік {int(max_row['Year'])}, тиждень {int(max_row['Week'])})")
        print(f"    Середнє = {prov_data.mean():.2f}, Медіана = {prov_data.median():.2f}")
        print()
    
    return pd.DataFrame(stats_list)

# Приклад: екстремуми для Харківської (20), Дніпропетровської (3) за 2000-2020
get_vhi_extremes(vhi_df, [20, 3], 2000, 2020)

### 5.4. Додаткові вибірки

#### 5.4.1. Роки з екстремальною посухою (VHI < 15) для вказаної області

In [ ]:
def get_drought_years(df, province_id, threshold=15):
    """
    Знаходить роки, коли VHI опускався нижче порогового значення (екстремальна посуха).
    """
    prov_data = df[df['Province_ID'] == province_id].copy()
    drought = prov_data[prov_data['VHI'] < threshold]
    
    name = UKR_INDEX.get(province_id, ("Невідома",))[0]
    
    if drought.empty:
        print(f"Для {name} (ID {province_id}) не знайдено тижнів з VHI < {threshold}")
        return pd.DataFrame()
    
    drought_years = drought['Year'].unique()
    print(f"Роки з екстремальною посухою (VHI < {threshold}) для {name}:")
    print(f"  {sorted([int(y) for y in drought_years if pd.notna(y)])}")
    print(f"  Всього тижнів з VHI < {threshold}: {len(drought)}")
    
    return drought[['Year', 'Week', 'VHI', 'Province_Name']].sort_values(['Year', 'Week']).reset_index(drop=True)

# Приклад: посухи для Херсонської області (21)
get_drought_years(vhi_df, 21)

#### 5.4.2. Порівняння середнього VHI між областями за рік

In [ ]:
def compare_provinces_vhi(df, year):
    """
    Порівнює середнє VHI всіх областей за вказаний рік.
    Повертає відсортований DataFrame.
    """
    year_data = df[df['Year'] == year]
    
    if year_data.empty:
        print(f"Даних за {year} рік не знайдено")
        return pd.DataFrame()
    
    comparison = year_data.groupby(['Province_ID', 'Province_Name'])['VHI'].agg(
        ['mean', 'min', 'max', 'count']
    ).round(2).reset_index()
    
    comparison.columns = ['ID', 'Область', 'Середнє VHI', 'Min VHI', 'Max VHI', 'Тижнів']
    comparison = comparison.sort_values('Середнє VHI', ascending=False).reset_index(drop=True)
    
    print(f"Порівняння областей за середнім VHI у {year} році:")
    return comparison

# Приклад: порівняння за 2020 рік
compare_provinces_vhi(vhi_df, 2020)

#### 5.4.3. Тренд VHI для області по роках (середнє за рік)

In [ ]:
def get_vhi_trend(df, province_id):
    """
    Показує тренд середнього VHI по роках для вказаної області.
    """
    prov_data = df[df['Province_ID'] == province_id]
    name = UKR_INDEX.get(province_id, ("Невідома",))[0]
    
    if prov_data.empty:
        print(f"Даних для {name} не знайдено")
        return pd.DataFrame()
    
    trend = prov_data.groupby('Year')['VHI'].mean().round(2).reset_index()
    trend.columns = ['Рік', 'Середнє VHI']
    
    print(f"Тренд VHI для {name} (ID {province_id}):")
    return trend

# Приклад: тренд для Полтавської області (16)
get_vhi_trend(vhi_df, 16)